# U04_C1 — The GenBank Flat File
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 4:** GenBank Sequence Database
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L7–L8, Practicals S4–S5 (GenBank & GenPept)
**Before this:** U03_C1

## 1. What you will learn
- **Dissect** each mandatory section of a GenBank flat file (LOCUS, DEFINITION, ACCESSION, VERSION, SOURCE, REFERENCE, FEATURES, ORIGIN) [LO2].
- **Interpret** GenBank location syntax including intervals, `join()`, and `complement()` [LO2].
- **Walk** the feature table to extract CDS coordinates and translate coding regions with Biopython [LO2].
- **Calculate** exact translated protein lengths directly from genomic and mRNA coordinates [LO2].


## 2. Biological question
When looking at the human beta-globin flat file (`NM_000518.5`), the mRNA is 626 bp long, but the mature beta-globin chain is only 146 or 147 amino acids. How does a bioinformatician read the flat file to locate the exact codon boundaries of the protein, understand where the untranslated regions end, and extract the translated peptide?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U04_C1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: Anatomy of a GenBank Flat File
### Theory
The GenBank flat file is structured into three main blocks:
1. **Header**:
   - `LOCUS`: Locus name, sequence length, molecule type (`mRNA`, `DNA`), topology (`linear`, `circular`), GenBank division (`PRI` = Primate, `ROD` = Rodent, `BCT` = Bacteria, `VRL` = Viral), and modification date.
   - `DEFINITION`: Free-text summary of organism, gene, and function.
   - `ACCESSION` & `VERSION`: Stable identifier and versioned accession (`NM_000518.5`).
   - `SOURCE` & `ORGANISM`: Formal taxonomic classification.
   - `REFERENCE`: Citations of primary literature supporting the sequence.
2. **Feature Table (`FEATURES`)**: Key-value annotations describing biological intervals.
3. **Sequence (`ORIGIN`)**: The raw sequence displayed in blocks of 10 nucleotides with 1-based line position numbers.


In [ ]:
# Worked Example: Inspecting Header metadata of HBB
from Bio import SeqIO

gb_path = DATA_DIR / "genbank" / "hbb_human.gb"
record = SeqIO.read(gb_path, "genbank")

print("LOCUS Name:    ", record.name)
print("Sequence Length:", len(record.seq), "bp")
print("Division:      ", record.annotations.get("data_file_division", "PRI"))
print("Topology:      ", record.annotations.get("topology", "linear"))
print("Accession:     ", record.id)
print("Definition:    ", record.description)


## 4. Concept 2: Location Syntax and the Features Table
### Theory
In the `FEATURES` block:
- Standard interval: `51..494` (nucleotides 51 through 494 inclusive, 1-based)
- Spliced exons: `join(51..142, 273..494)` indicates exons spliced together in mRNA
- Opposite strand: `complement(51..494)` indicates coding on the reverse strand
- Fuzzy boundaries: `<1..500` indicates incomplete 5' end; `100..>600` indicates incomplete 3' end

In Biopython, coordinates are automatically converted into 0-based half-open Python slices (`[start:end]`).


In [ ]:
# Worked Example: Walking the features table and isolating the CDS
cds_features = [f for f in record.features if f.type == "CDS"]
assert len(cds_features) == 1, "Expected single CDS in HBB mRNA"
cds_feat = cds_features[0]

print("CDS Location string in GenBank: 51..494 (1-based)")
print(f"Biopython 0-based slice:        {cds_feat.location.start}:{cds_feat.location.end}")
print(f"CDS nucleotide length:          {cds_feat.location.end - cds_feat.location.start} bp")

# Expected protein length: (CDS length - 3 stop nucleotides) / 3
cds_nt_len = cds_feat.location.end - cds_feat.location.start
expected_aa_len = (cds_nt_len - 3) // 3
print(f"Calculated protein length:      {expected_aa_len} amino acids")


In [ ]:
# Try it: Extract and translate the CDS sequence using Biopython
cds_seq = cds_feat.extract(record.seq)
translated = cds_seq.translate(to_stop=True)

print("CDS start codon:", cds_seq[:3])
print("CDS stop codon: ", cds_seq[-3:])
print("Translated protein length:", len(translated), "aa")
print("Official /translation in flat file:", cds_feat.qualifiers["translation"][0])
print("Matches official annotation?", str(translated) == cds_feat.qualifiers["translation"][0])


**Common mistakes:**
- Forgetting that CDS end coordinates include the 3-nucleotide stop codon (TAA, TAG, TGA).
- Confusing 1-based GenBank coordinates (`51..494` -> length = $494 - 51 + 1 = 444$) with Python 0-based slices (`[50:494]` -> length = $494 - 50 = 444$).


## 5. Compare with the Web Tool
In Practical S4, we scrolled down the flat file on NCBI Nucleotide:
- CDS: `51..494`
- `/gene="HBB"`
- `/codon_start=1`
- `/protein_id="NP_000509.1"`
- `/translation="MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPK..."`
Our programmatic extraction produces the identical 147 amino acid sequence and verified coordinate calculation.


## 6. Summary
- The GenBank flat file contains LOCUS, DEFINITION, ACCESSION, FEATURES, and ORIGIN blocks.
- Location syntax indicates coordinates (`51..494`), spliced joints (`join()`), and opposite strand (`complement()`).
- `feature.extract(record.seq)` splices exons and extracts the exact sub-sequence.
- Coding sequence length in nucleotides (including stop codon) divided by 3 gives $(N_{\text{aa}} + 1)$.


## 7. Next Steps
- Learn about format conversions in `U04_C2_Formats_and_Conversion.ipynb`.
- Solve feature extraction exercises in `U04_P_GenBank_and_Formats.ipynb`.
